# Final experiment: tuning → FINAL TEST → explanations → app model

Run top to bottom. Each section saves its output, so after a restart you can re-run without redoing slow steps.

| Section | Time | Saves |
|---|---|---|
| 1. Data + features + Dixon–Coles | ~1 min | `poisson_probs.csv` |
| 2. Tuning (loads saved results if they exist) | 0 or ~15 min | `tuning_results.csv` |
| 3. **FINAL TEST, once** | ~1 min | `results_table.csv`, `test_predictions.npz`, models |
| 4. Bootstrap (real or luck?) | ~10 s | `rq_bootstrap.csv` |
| 5. Charts + tables | ~1 min | `results/figures/*.png` |
| 6. App model | ~1 min | `results/models/app_model.joblib` |

**Rule:** never change settings after looking at Section 3. Re-running with the same settings is fine (same numbers).

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("working in:", os.getcwd())

## 1. Data, features, Dixon–Coles probabilities

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from src.load_data import read_matches
from src.features import build_features, SETUPS
from src.poisson import walk_forward_probs, fill_missing_probs

m = read_matches()
f = build_features(m)

probs_path = Path("data/processed/poisson_probs.csv")
probs = pd.read_csv(probs_path) if probs_path.exists() else None
if probs is None or "pois_rho" not in probs or probs["pois_rho"].abs().sum() == 0:
    print("computing Dixon-Coles probabilities (~1 min)...")
    probs = walk_forward_probs(m, xi=0.0019, dixon_coles=True)
    probs.to_csv(probs_path, index=False)
f = f.merge(fill_missing_probs(probs), on="match_id", how="left")
print(len(f), "matches | avg rho:", round(probs.pois_rho.mean(), 3))

## 2. Tuning on 5 seasons (2018/19–2022/23)
Loads `results/tuning_results.csv` if it exists. The second cell tries the extra (simpler) settings and keeps any that are better; it's safe to run again.

In [ ]:
from src.models import tune_rolling, load_tuned, extend_tuning
tune_path = Path("results/tuning_results.csv")
if tune_path.exists():
    tuned = load_tuned(tune_path)
    print("loaded saved tuning results")
else:
    print("tuning (~15 min)...")
    tuned = tune_rolling(f, SETUPS)
    tuned.to_csv(tune_path, index=False)
tuned.pivot(index="model", columns="setup", values="rps").round(4)

In [ ]:
tuned = extend_tuning(f, tuned, SETUPS)
tuned.to_csv(tune_path, index=False)
print("FROZEN SETTINGS (copy these into the Results Log):")
tuned[["model", "setup", "params", "accuracy", "rps"]]

## 3. ⚠️ FINAL TEST: 2023/24 + 2024/25, run once
Trains every tuned model on 2010/11–2022/23 and scores it on the 760 test matches. It refuses to run a second time. If you need to reproduce it after a crash (same settings), use `force=True`.

In [ ]:
from src.models import run_final_test
final = run_final_test(f, tuned, SETUPS)
final.sort_values("rps")[["model", "setup", "accuracy", "macro_F1", "brier", "rps"]]

## 4. Are the differences real or luck? (bootstrap, 1,000 resamples)
`diff` < 0 means the newer setup is better. `clear = True` means the whole 95% range is on one side of zero.

In [ ]:
from src import analysis as an
final = pd.read_csv("results/results_table.csv")
y, match_id, preds = an.load_predictions()
rq = an.research_question_table(y, preds)
rq.to_csv("results/rq_bootstrap.csv", index=False)
rq

## 5. Charts and tables for the report

In [ ]:
print(an.plot_setup_comparison(final))

ml = final[final.model.isin(["LR", "RF", "XGB"])].sort_values("rps")
best = ml.iloc[0]
best_key = f"{best.model}_{best.setup}"
print("best model:", best_key, "| rps", best.rps, "| accuracy", best.accuracy)

test = f.set_index("match_id").loc[match_id].reset_index()
print(an.plot_confusion(y, preds[best_key], f"Confusion matrix: {an.MODEL_NAMES[best.model]}, setup {best.setup}")[0])
draws, draw_summary = an.draw_analysis(y, preds[best_key], test.elo_diff)
print(an.plot_draws(draws))
print(draw_summary)
draws

In [ ]:
print(an.plot_calibration(y, {
    f"{an.MODEL_NAMES[best.model]} ({best.setup})": (preds[best_key], an.COLORS[best.model]),
    "Bookmaker": (preds["Bookmaker (Bet365)"], an.GREY),
}))

import joblib
lr = joblib.load("results/models/LR_D.joblib")
coefs = an.lr_coefficients(lr, SETUPS["D"])
print(an.plot_lr_coefficients(coefs))
coefs.head(12).round(3)

In [ ]:
xgb = joblib.load("results/models/XGB_C.joblib")
imp, path, rank = an.shap_importance(xgb, test[SETUPS["C"]])
print(path)
print("head-to-head rank (out of", len(rank), "):", rank.get("h2h_score"), "| h2h_n:", rank.get("h2h_n"))
imp.head(12).round(4)

In [ ]:
path, ha = an.plot_home_advantage(m)
print(path)

from src.predict import poisson_for_fixture
home, away = "Arsenal", "Chelsea"
(pH, pD, pA), grid, xg, rho = poisson_for_fixture(m, home, away, m.date.max() + pd.Timedelta(days=7))
print(an.plot_scoreline_grid(grid, home, away))
print(f"{home} {pH:.0%} / draw {pD:.0%} / {away} {pA:.0%} | xG {xg[0]:.2f}-{xg[1]:.2f} | rho {rho:.3f}")
ha.tail(8)

## 6. App model (trained on all seasons 2010/11–2024/25)
Uses the tuned LR setup D settings. If your final results pick a different best model, change `model_name` / `setup`.

In [ ]:
from src.predict import train_app_model, predict_fixture
row = tuned[(tuned.model == "LR") & (tuned.setup == "D")].iloc[0]
train_app_model("LR", "D", row.params)
r = predict_fixture("Arsenal", "Chelsea", m)
print(r["source"], [round(p, 3) for p in r["probs"]], r["top_scores"][:3])

## Done
Push everything:
```
git add results data notebooks
git commit -m "Final test, figures, app model"
git push
```
Then run the app: `streamlit run app/app.py`.